# Домашнее задание 1.2

In [1]:
!pwd

/home/jovyan/work/hw_1_2


In [2]:
!ls -la

total 36
drwxrwxrwx 1 root root  4096 Sep 30 21:43 .
drwxrwxrwx 1 root root  4096 Sep 30 12:43 ..
drwxrwxrwx 1 root root  4096 Sep 30 10:41 data
-rw-r--r-- 1 root root 33113 Sep 30 21:43 hw_1_2.ipynb
drwxr-xr-x 1 root root  4096 Sep 30 12:36 .ipynb_checkpoints


In [3]:
import sys
import pyspark

print(sys.version)
print(pyspark.__version__)

3.8.20 | packaged by conda-forge | (default, Sep 30 2024, 17:52:49) 
[GCC 13.3.0]
3.5.0


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .master("spark://spark-master:7077") \
    .appName("hw_1_2") \
    .getOrCreate()

In [5]:
sc = spark.sparkContext
sc.master

'spark://spark-master:7077'

## Задача 1

In [6]:
!hdfs dfs -ls /

Found 2 items
drwxr-xr-x   - hadoop users          0 2026-09-30 17:47 /hw_1_2
drwxr-xr-x   - hadoop users          0 2026-09-30 15:14 /user


!head -n 3 data/bible.txt

In [7]:
!hdfs dfs -mkdir -p /hw_1_2/task1/input

In [8]:
!hdfs dfs -put -f data/bible.txt /hw_1_2/task1/input/

In [9]:
!hdfs dfs -ls /hw_1_2/task1/input

Found 1 items
-rw-r--r--   1 hadoop users    4047392 2026-09-30 21:44 /hw_1_2/task1/input/bible.txt


#### RDD

In [10]:
rdd = sc.textFile("hdfs://namenode:8020/hw_1_2/task1/input/bible.txt")

In [11]:
words = rdd.flatMap(lambda line: line.split())

In [12]:
long_words = words.filter(lambda word: len(word) > 4)

In [13]:
count = long_words.count()

In [14]:
count

266384

*Это число совпадает с числом полученным благодаря MR*

#### DataFrame API

In [15]:
from pyspark.sql.functions import split, explode, length, col

text = spark.read.text("hdfs://namenode:8020/hw_1_2/task1/input/bible.txt")

In [16]:
words = text.select(
    explode(split(col("value"), " ")).alias("word")
)

In [17]:
filtered = words.filter(length(col("word")) > 4)

In [18]:
count = filtered.count()

In [19]:
count

266384

*Результат как видим правильный*

### Задача 2

#### RDD

In [20]:
!hdfs dfs -mkdir -p /hw_1_2/task2/input

In [21]:
!hdfs dfs -put -f data/Poseschenia_sai_774_tov.csv /hw_1_2/task2/input/

In [22]:
!hdfs dfs -ls /hw_1_2/task2/input

Found 1 items
-rw-r--r--   1 hadoop users   36443383 2026-09-30 21:44 /hw_1_2/task2/input/Poseschenia_sai_774_tov.csv


In [23]:
import heapq

In [24]:
lines = sc.textFile("/hw_1_2/task2/input/Poseschenia_sai_774_tov.csv")

In [25]:
def parse_line(line):
    try:
        site, datetime_value = line.strip().split(";", 1)

        date = datetime_value.strip().split()[0]

        if not site or not date:
            return []

        return [(date, site)]

    except (ValueError, IndexError):
        return []


rows = lines.flatMap(parse_line)

rows.take(5)

[('2024-05-28', 'https://gonzales-bautista.com/'),
 ('2024-05-26', 'https://gonzales-bautista.com/'),
 ('2024-05-31', 'https://www.davis-berg.com/'),
 ('2024-05-29', 'https://www.montoya.com/'),
 ('2024-06-01', 'https://gonzales-bautista.com/')]

In [26]:
visits = rows.map(lambda date_site: (date_site, 1))

site_counts = visits.reduceByKey(lambda left, right: left + right)

site_counts.take(5)

[(('2024-05-28', 'https://gonzales-bautista.com/'), 368),
 (('2024-05-26', 'https://gonzales-bautista.com/'), 335),
 (('2024-05-31', 'https://www.davis-berg.com/'), 38),
 (('2024-06-01', 'https://gonzales-bautista.com/'), 379),
 (('2024-05-27', 'https://www.davis-berg.com/'), 38)]

In [27]:
date_candidates = site_counts.map(
    lambda item: (item[0][0], (item[1], item[0][1]))
)

date_candidates.take(5)

[('2024-05-28', (368, 'https://gonzales-bautista.com/')),
 ('2024-05-26', (335, 'https://gonzales-bautista.com/')),
 ('2024-05-31', (38, 'https://www.davis-berg.com/')),
 ('2024-06-01', (379, 'https://gonzales-bautista.com/')),
 ('2024-05-27', (38, 'https://www.davis-berg.com/'))]

In [28]:
TOP_N = 5

def add_to_top5(top, candidate):
    top.append(candidate)

    top.sort(key=lambda item: (-item[0], item[1]))

    del top[TOP_N:]

    return top

def create_combiner(candidate):
    return [candidate]

def merge_value(top, candidate):
    return add_to_top5(top, candidate)

def merge_combiners(top_left, top_right):
    for candidate in top_right:
        add_to_top5(top_left, candidate)

    return top_left

In [29]:
top5_by_date = date_candidates.combineByKey(
    create_combiner,
    merge_value,
    merge_combiners
)

top5_by_date.take(5)

[('2024-05-26',
  [(335, 'https://gonzales-bautista.com/'),
   (235, 'http://smith.com/'),
   (221, 'https://www.smith.com/'),
   (212, 'http://www.smith.com/'),
   (212, 'https://smith.com/')]),
 ('2024-05-30',
  [(353, 'https://gonzales-bautista.com/'),
   (246, 'https://smith.com/'),
   (239, 'https://www.smith.com/'),
   (229, 'http://smith.com/'),
   (225, 'http://www.smith.com/')]),
 ('2024-05-29',
  [(402, 'https://gonzales-bautista.com/'),
   (242, 'https://www.smith.com/'),
   (223, 'http://www.smith.com/'),
   (220, 'https://smith.com/'),
   (206, 'http://smith.com/')]),
 ('2024-05-28',
  [(368, 'https://gonzales-bautista.com/'),
   (256, 'https://smith.com/'),
   (251, 'https://www.smith.com/'),
   (224, 'http://smith.com/'),
   (204, 'http://www.smith.com/')]),
 ('2024-06-01',
  [(379, 'https://gonzales-bautista.com/'),
   (232, 'https://www.smith.com/'),
   (226, 'https://smith.com/'),
   (195, 'https://johnson.com/'),
   (191, 'http://smith.com/')])]

In [30]:
def format_top5(item):
    date, top_sites = item

    result = []

    for rank, (count, site) in enumerate(top_sites, start=1):
        result.append(f"{date}\t{rank}\t{site}\t{count}")

    return result


result_lines = (
    top5_by_date
    .sortByKey()
    .flatMap(format_top5)
)

result_lines.take(20)

['2024-05-26\t1\thttps://gonzales-bautista.com/\t335',
 '2024-05-26\t2\thttp://smith.com/\t235',
 '2024-05-26\t3\thttps://www.smith.com/\t221',
 '2024-05-26\t4\thttp://www.smith.com/\t212',
 '2024-05-26\t5\thttps://smith.com/\t212',
 '2024-05-27\t1\thttps://gonzales-bautista.com/\t376',
 '2024-05-27\t2\thttps://www.smith.com/\t270',
 '2024-05-27\t3\thttps://smith.com/\t236',
 '2024-05-27\t4\thttp://smith.com/\t215',
 '2024-05-27\t5\thttp://www.smith.com/\t208',
 '2024-05-28\t1\thttps://gonzales-bautista.com/\t368',
 '2024-05-28\t2\thttps://smith.com/\t256',
 '2024-05-28\t3\thttps://www.smith.com/\t251',
 '2024-05-28\t4\thttp://smith.com/\t224',
 '2024-05-28\t5\thttp://www.smith.com/\t204',
 '2024-05-29\t1\thttps://gonzales-bautista.com/\t402',
 '2024-05-29\t2\thttps://www.smith.com/\t242',
 '2024-05-29\t3\thttp://www.smith.com/\t223',
 '2024-05-29\t4\thttps://smith.com/\t220',
 '2024-05-29\t5\thttp://smith.com/\t206']

In [31]:
!hdfs dfs -rm -r -f /hw_1_2/task2/output/rdd_top5

Deleted /hw_1_2/task2/output/rdd_top5


In [32]:
result_lines.saveAsTextFile("/hw_1_2/task2/output/rdd_top5")

In [33]:
!hdfs dfs -cat /hw_1_2/task2/output/rdd_top5/part-*

2024-05-26	1	https://gonzales-bautista.com/	335
2024-05-26	2	http://smith.com/	235
2024-05-26	3	https://www.smith.com/	221
2024-05-26	4	http://www.smith.com/	212
2024-05-26	5	https://smith.com/	212
2024-05-27	1	https://gonzales-bautista.com/	376
2024-05-27	2	https://www.smith.com/	270
2024-05-27	3	https://smith.com/	236
2024-05-27	4	http://smith.com/	215
2024-05-27	5	http://www.smith.com/	208
2024-05-28	1	https://gonzales-bautista.com/	368
2024-05-28	2	https://smith.com/	256
2024-05-28	3	https://www.smith.com/	251
2024-05-28	4	http://smith.com/	224
2024-05-28	5	http://www.smith.com/	204
2024-05-29	1	https://gonzales-bautista.com/	402
2024-05-29	2	https://www.smith.com/	242
2024-05-29	3	http://www.smith.com/	223
2024-05-29	4	https://smith.com/	220
2024-05-29	5	http://smith.com/	206
2024-05-30	1	https://gonzales-bautista.com/	353
2024-05-30	2	https://smith.com/	246
2024-05-30	3	https://www.smith.com/	239
2024-05-30	4	http://smith.com/	229
2024-05-30	5	http://www.smith.com/	225
2024-05-31

*Можно залезть в ноутбук дз_1_1 и убедиться, что результат идентичный*

#### DataFrame API

In [34]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

INPUT_PATH = "hdfs://namenode:8020/hw_1_2/task2/input/Poseschenia_sai_774_tov.csv"

In [35]:
raw_df = (
    spark.read
    .option("header", "false")
    .option("sep", ";")
    .csv(INPUT_PATH)
    .toDF("site", "event_time")
)

raw_df.printSchema()
raw_df.show(5, truncate=False)

root
 |-- site: string (nullable = true)
 |-- event_time: string (nullable = true)

+------------------------------+--------------------------+
|site                          |event_time                |
+------------------------------+--------------------------+
|https://gonzales-bautista.com/|2024-05-28 05:58:33.853479|
|https://gonzales-bautista.com/|2024-05-26 03:44:36.853479|
|https://www.davis-berg.com/   |2024-05-31 17:22:01.853479|
|https://www.montoya.com/      |2024-05-29 13:41:35.853479|
|https://gonzales-bautista.com/|2024-06-01 06:49:16.853479|
+------------------------------+--------------------------+
only showing top 5 rows



In [36]:
visits_df = (
    raw_df
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss.SSSSSS")
    )
    .withColumn("date", F.to_date("event_timestamp"))
    .filter(
        F.col("site").isNotNull()
        & F.col("date").isNotNull()
        & (F.length(F.trim(F.col("site"))) > 0)
    )
    .select("date", "site")
)

visits_df.show(5, truncate=False)

+----------+------------------------------+
|date      |site                          |
+----------+------------------------------+
|2024-05-28|https://gonzales-bautista.com/|
|2024-05-26|https://gonzales-bautista.com/|
|2024-05-31|https://www.davis-berg.com/   |
|2024-05-29|https://www.montoya.com/      |
|2024-06-01|https://gonzales-bautista.com/|
+----------+------------------------------+
only showing top 5 rows



In [37]:
site_counts_df = (
    visits_df
    .groupBy("date", "site")
    .count()
    .withColumnRenamed("count", "visits")
)

site_counts_df.show(10, truncate=False)

+----------+-----------------------------+------+
|date      |site                         |visits|
+----------+-----------------------------+------+
|2024-05-29|http://hawkins-harris.com/   |1     |
|2024-05-31|http://www.morrow.org/       |2     |
|2024-05-30|https://www.bauer.com/       |12    |
|2024-05-26|https://kent.com/            |5     |
|2024-05-28|https://www.gould.com/       |1     |
|2024-05-31|https://le-meyer.com/        |1     |
|2024-05-28|https://www.lynch-cook.com/  |3     |
|2024-06-01|https://rogers-henderson.com/|1     |
|2024-05-28|http://norris.com/           |10    |
|2024-05-28|http://stewart.biz/          |6     |
+----------+-----------------------------+------+
only showing top 10 rows



In [38]:
window_spec = (
    Window
    .partitionBy("date")
    .orderBy(
        F.col("visits").desc(),
        F.col("site").asc()
    )
)

top5_df = (
    site_counts_df
    .withColumn("rank", F.row_number().over(window_spec))
    .filter(F.col("rank") <= 5)
    .orderBy("date", "rank")
)

top5_df.show(50, truncate=False)

+----------+------------------------------+------+----+
|date      |site                          |visits|rank|
+----------+------------------------------+------+----+
|2024-05-26|https://gonzales-bautista.com/|335   |1   |
|2024-05-26|http://smith.com/             |235   |2   |
|2024-05-26|https://www.smith.com/        |221   |3   |
|2024-05-26|http://www.smith.com/         |212   |4   |
|2024-05-26|https://smith.com/            |212   |5   |
|2024-05-27|https://gonzales-bautista.com/|376   |1   |
|2024-05-27|https://www.smith.com/        |270   |2   |
|2024-05-27|https://smith.com/            |236   |3   |
|2024-05-27|http://smith.com/             |215   |4   |
|2024-05-27|http://www.smith.com/         |208   |5   |
|2024-05-28|https://gonzales-bautista.com/|368   |1   |
|2024-05-28|https://smith.com/            |256   |2   |
|2024-05-28|https://www.smith.com/        |251   |3   |
|2024-05-28|http://smith.com/             |224   |4   |
|2024-05-28|http://www.smith.com/         |204  

In [39]:
!hdfs dfs -rm -r -f /hw_1_2/task2/output/dataframe_top5

Deleted /hw_1_2/task2/output/dataframe_top5


In [40]:
(
    top5_df
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("sep", "\t")
    .csv("hdfs://namenode:8020/hw_1_2/task2/output/dataframe_top5")
)

In [41]:
!hdfs dfs -cat /hw_1_2/task2/output/dataframe_top5/part-*

date	site	visits	rank
2024-05-26	https://gonzales-bautista.com/	335	1
2024-05-26	http://smith.com/	235	2
2024-05-26	https://www.smith.com/	221	3
2024-05-26	http://www.smith.com/	212	4
2024-05-26	https://smith.com/	212	5
2024-05-27	https://gonzales-bautista.com/	376	1
2024-05-27	https://www.smith.com/	270	2
2024-05-27	https://smith.com/	236	3
2024-05-27	http://smith.com/	215	4
2024-05-27	http://www.smith.com/	208	5
2024-05-28	https://gonzales-bautista.com/	368	1
2024-05-28	https://smith.com/	256	2
2024-05-28	https://www.smith.com/	251	3
2024-05-28	http://smith.com/	224	4
2024-05-28	http://www.smith.com/	204	5
2024-05-29	https://gonzales-bautista.com/	402	1
2024-05-29	https://www.smith.com/	242	2
2024-05-29	http://www.smith.com/	223	3
2024-05-29	https://smith.com/	220	4
2024-05-29	http://smith.com/	206	5
2024-05-30	https://gonzales-bautista.com/	353	1
2024-05-30	https://smith.com/	246	2
2024-05-30	https://www.smith.com/	239	3
2024-05-30	http://smith.com/	229	4
2024-05-30	http://www.smith

*Результат идентичный*